In [ ]:
# =============================================================================
# 📦 IMPORTS & CONFIGURATION
# =============================================================================
import os
import json
import random
import gc
import shutil
from pathlib import Path
from typing import List, Tuple, Dict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from scipy.special import softmax
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression  # Pour calibration isotonique
from sklearn.feature_selection import mutual_info_classif  # Pour feature selection
import lightgbm as lgb  # Pour adversarial validation
from xgboost import XGBClassifier
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)
import warnings
warnings.filterwarnings('ignore')

# Caches pour limiter écriture disque
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf/transformers"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf/datasets"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
Path("/tmp/hf/transformers").mkdir(parents=True, exist_ok=True)
Path("/tmp/hf/datasets").mkdir(parents=True, exist_ok=True)

# =============================================================================
# 🧹 FONCTIONS DE GESTION MÉMOIRE
# =============================================================================
def clean_memory(verbose: bool = False):
    """Libère la mémoire GPU et RAM de manière agressive."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        if verbose:
            allocated = torch.cuda.memory_allocated() / 1e9
            reserved = torch.cuda.memory_reserved() / 1e9
            print(f"   🧹 GPU: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

def clean_tmp_folder(folder: str = "/tmp/transformer_ultimate"):
    """Supprime les fichiers temporaires d'entraînement."""
    tmp_path = Path(folder)
    if tmp_path.exists():
        shutil.rmtree(tmp_path, ignore_errors=True)
        
def print_memory_status():
    """Affiche l'état de la mémoire GPU."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        total = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"📊 GPU Memory: {allocated:.2f}GB / {total:.2f}GB (reserved: {reserved:.2f}GB)")

print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"PyTorch version: {torch.__version__}")
print_memory_status()

/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/kaggle-env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch CUDA available: True
GPU name: NVIDIA RTX 4000 Ada Generation
PyTorch version: 2.8.0+cu128
📊 GPU Memory: 0.00GB / 20.97GB (reserved: 0.00GB)


In [2]:
@dataclass
class Config:
    """Configuration centralisée pour le pipeline - VERSION OPTIMISÉE."""
    # Modèle
    MODEL_NAME: str = "cardiffnlp/twitter-xlm-roberta-base"
    MAX_LEN: int = 224  # ⬆️ Augmenté de 192 à 224 (moins de troncature)
    
    # Entraînement Transformer
    N_SPLITS: int = 5   # ⬆️ Augmenté de 4 à 5 (80% train au lieu de 75%)
    EPOCHS: int = 5     # ⬆️ Augmenté de 4 à 5 (meilleure convergence)
    BATCH_SIZE: int = 16
    GRAD_ACC: int = 2   # effective batch = 32
    LR: float = 2e-5
    WEIGHT_DECAY: float = 0.01
    WARMUP_RATIO: float = 0.1
    
    # Multi-seed - Plus de seeds = plus de diversité = meilleur ensemble
    SEEDS: List[int] = None  # ⬆️ [42, 1234, 2024, 0] - 4 seeds au lieu de 2
    
    # XGBoost GPU
    XGB_MAX_DEPTH: int = 10
    XGB_N_ESTIMATORS: int = 2000  # ⬆️ Augmenté de 1500 à 2000
    XGB_LR: float = 0.02          # ⬇️ Réduit pour plus d'arbres
    XGB_SUBSAMPLE: float = 0.8
    XGB_COLSAMPLE: float = 0.8
    XGB_MIN_CHILD_WEIGHT: int = 1
    
    # Pseudo-labeling - Seuils modérés
    PSEUDO_THR_HIGH: float = 0.85
    PSEUDO_THR_LOW: float = 0.15
    PSEUDO_EPOCHS: int = 2
    PSEUDO_LR: float = 1e-5
    
    # Chemins
    PROJECT_ROOT: Path = None
    
    def __post_init__(self):
        if self.SEEDS is None:
            self.SEEDS = [42, 1234, 2024, 0]  # ⬆️ 4 seeds pour plus de diversité
        if self.PROJECT_ROOT is None:
            self.PROJECT_ROOT = Path.cwd()
            if not (self.PROJECT_ROOT / "data").exists():
                self.PROJECT_ROOT = self.PROJECT_ROOT.parent

CFG = Config()
DATA_DIR = CFG.PROJECT_ROOT / "data"
SUBMISSION_DIR = CFG.PROJECT_ROOT / "submission"
MODEL_DIR = CFG.PROJECT_ROOT / "models/transformer_ultimate"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Config OPTIMISÉE loaded")
print(f"   MAX_LEN={CFG.MAX_LEN}, EPOCHS={CFG.EPOCHS}, N_SPLITS={CFG.N_SPLITS}")
print(f"   SEEDS={CFG.SEEDS} ({len(CFG.SEEDS)} seeds)")
print(f"   XGB: max_depth={CFG.XGB_MAX_DEPTH}, n_estimators={CFG.XGB_N_ESTIMATORS}, lr={CFG.XGB_LR}")
print(f"   Pseudo-labeling: THR_HIGH={CFG.PSEUDO_THR_HIGH}, THR_LOW={CFG.PSEUDO_THR_LOW}")
print(f"\n⏱️ Temps estimé: ~{len(CFG.SEEDS) * CFG.N_SPLITS * 15}min pour Transformer + ~10min XGB")

✅ Config OPTIMISÉE loaded
   MAX_LEN=224, EPOCHS=5, N_SPLITS=5
   SEEDS=[42, 1234, 2024, 0] (4 seeds)
   XGB: max_depth=10, n_estimators=2000, lr=0.02
   Pseudo-labeling: THR_HIGH=0.85, THR_LOW=0.15

⏱️ Temps estimé: ~300min pour Transformer + ~10min XGB


In [3]:
def set_seed(seed: int):
    """Fixe toutes les graines pour la reproductibilité."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(CFG.SEEDS[0])

## 📊 Chargement et Préparation des Données

In [4]:
# =============================================================================
# FONCTIONS DE PREPROCESSING TEXTE
# =============================================================================

def parse_source(source_html: str) -> str:
    """Extrait le type de source depuis le HTML."""
    if not isinstance(source_html, str):
        return "unknown"
    src = source_html.lower()
    if "iphone" in src:
        return "iphone"
    if "android" in src:
        return "android"
    if "tweetdeck" in src:
        return "tweetdeck"
    if "web" in src or "browser" in src:
        return "web"
    if any(x in src for x in ["buffer", "hootsuite", "socialflow", "sprout", "dlvr.it", "ifttt", "publicize"]):
        return "bot"
    return "other"

def extract_text(row: pd.Series) -> str:
    """Extrait le texte complet du tweet."""
    if isinstance(row.get("extended_tweet"), dict):
        full = row["extended_tweet"].get("full_text")
        if full:
            return str(full)
    if isinstance(row.get("full_text"), str) and row["full_text"]:
        return str(row["full_text"])
    if isinstance(row.get("text"), str):
        return str(row["text"])
    return ""

def build_input_text(row: pd.Series) -> str:
    """Construit le texte d'entrée enrichi avec métadonnées."""
    user = row.get("user") or {}
    text = extract_text(row)
    desc = (user.get("description") or "")[:200]  # Plus de description
    location = (user.get("location") or "")[:80]
    source = parse_source(row.get("source", ""))
    
    # Features numériques normalisées
    statuses = user.get("statuses_count") or 0
    favourites = user.get("favourites_count") or 0
    listed = user.get("listed_count") or 0
    
    # Features binaires
    has_url = 1 if user.get("url") else 0
    has_banner = 1 if user.get("profile_banner_url") else 0
    is_reply = 1 if row.get("in_reply_to_status_id") is not None else 0
    is_retweet = 1 if row.get("retweeted_status") is not None else 0
    is_quote = 1 if row.get("is_quote_status", False) else 0
    
    # Ratios (bons indicateurs d'influence)
    ratio_listed = listed / max(1, statuses) * 1000
    
    meta = (f"device={source} statuses={statuses} favs={favourites} listed={listed} "
            f"reply={is_reply} retweet={is_retweet} quote={is_quote} url={has_url} "
            f"banner={has_banner} ratio={ratio_listed:.1f}")
    
    return f"{text} [DESC] {desc} [LOC] {location} [META] {meta}"

def pseudo_user_id(row: pd.Series) -> str:
    """Crée un pseudo-ID utilisateur pour le groupement CV."""
    user = row.get("user") or {}
    key = "|".join([
        str(user.get("description", ""))[:64],
        str(user.get("profile_image_url_https", "")),
        str(user.get("profile_banner_url", "")),
        str(user.get("statuses_count", 0)),
    ])
    return str(abs(hash(key)) % (10 ** 12))

In [5]:
# Chargement des données brutes
print("📂 Chargement des données...")
train_raw = pd.read_json(DATA_DIR / "train.jsonl", lines=True)
test_raw = pd.read_json(DATA_DIR / "kaggle_test.jsonl", lines=True)

# Preprocessing
train_raw["text_input"] = train_raw.apply(build_input_text, axis=1)
test_raw["text_input"] = test_raw.apply(build_input_text, axis=1)
train_raw["group"] = train_raw.apply(pseudo_user_id, axis=1)
test_raw["group"] = test_raw.apply(pseudo_user_id, axis=1)

labels = train_raw["label"].astype(int).to_numpy()
groups = train_raw["group"].to_numpy()
test_ids = test_raw["challenge_id"].astype(int).to_numpy()

print(f"\n📊 Distribution des labels:")
print(train_raw["label"].value_counts(normalize=True))
print(f"\n✅ Train: {len(train_raw):,} tweets | Test: {len(test_raw):,} tweets")
print(f"   Groupes uniques train: {train_raw['group'].nunique():,}")

📂 Chargement des données...

📊 Distribution des labels:
label
0    0.533677
1    0.466323
Name: proportion, dtype: float64

✅ Train: 154,914 tweets | Test: 103,380 tweets
   Groupes uniques train: 153,261

📊 Distribution des labels:
label
0    0.533677
1    0.466323
Name: proportion, dtype: float64

✅ Train: 154,914 tweets | Test: 103,380 tweets
   Groupes uniques train: 153,261


In [6]:
# Chargement des features engineered pour XGBoost
print("📂 Chargement des features...")
X_feat = np.load(DATA_DIR / "features/X_train_features.npy")
X_test_feat = np.load(DATA_DIR / "features/X_kaggle_features.npy")

print(f"✅ Features train: {X_feat.shape} | Features test: {X_test_feat.shape}")

📂 Chargement des features...
✅ Features train: (154914, 84) | Features test: (103380, 84)


In [7]:
# Tokenisation
print(f"🔤 Tokenisation avec {CFG.MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)

train_ds = Dataset.from_pandas(train_raw[["text_input", "label", "group"]])
test_ds = Dataset.from_pandas(test_raw[["challenge_id", "text_input", "group"]])

train_remove_cols = [c for c in ["text_input", "group", "__index_level_0__"] if c in train_ds.column_names]
test_remove_cols = [c for c in ["text_input", "group", "challenge_id", "__index_level_0__"] if c in test_ds.column_names]

def tokenize_fn(batch):
    return tokenizer(batch["text_input"], truncation=True, max_length=CFG.MAX_LEN)

train_tokenized = train_ds.map(tokenize_fn, batched=True, remove_columns=train_remove_cols)
test_tokenized = test_ds.map(tokenize_fn, batched=True, remove_columns=test_remove_cols)

collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print(f"✅ Tokenisation terminée (MAX_LEN={CFG.MAX_LEN})")

🔤 Tokenisation avec cardiffnlp/twitter-xlm-roberta-base...


Map: 100%|██████████| 103380/103380 [00:18<00:00, 5458.97 examples/s]

✅ Tokenisation terminée (MAX_LEN=224)


## 🎯 Phase 1: Transformer Multi-Seed CV

In [8]:
def compute_metrics(eval_pred):
    """Calcule l'accuracy pour l'évaluation."""
    logits, labels_arr = eval_pred
    preds = logits.argmax(-1)
    return {"accuracy": accuracy_score(labels_arr, preds)}

def train_transformer_cv(seed: int, verbose: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    """Entraîne le Transformer en CV avec une seed donnée.
    
    Returns:
        oof_probs: Out-of-fold predictions sur train
        test_probs: Predictions moyennées sur test
    """
    set_seed(seed)
    if verbose:
        print(f"\n{'='*60}")
        print(f"🌱 SEED = {seed}")
        print(f"{'='*60}")
    
    skf = StratifiedGroupKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=seed)
    
    oof_probs = np.zeros(len(train_tokenized))
    test_probs = np.zeros((len(test_tokenized), 2))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels)), labels, groups)):
        if verbose:
            print(f"\n----- Fold {fold+1}/{CFG.N_SPLITS} -----")
        
        train_split = train_tokenized.select(train_idx.tolist())
        val_split = train_tokenized.select(val_idx.tolist())
        
        # ignore_mismatched_sizes supprime le warning sur l'initialisation du classifier
        model = AutoModelForSequenceClassification.from_pretrained(
            CFG.MODEL_NAME, 
            num_labels=2,
            ignore_mismatched_sizes=True
        )
        
        out_dir = Path(f"/tmp/transformer_ultimate/seed{seed}/fold{fold}")
        out_dir.mkdir(parents=True, exist_ok=True)
        
        args = TrainingArguments(
            output_dir=str(out_dir),
            num_train_epochs=CFG.EPOCHS,
            per_device_train_batch_size=CFG.BATCH_SIZE,
            per_device_eval_batch_size=CFG.BATCH_SIZE * 2,
            learning_rate=CFG.LR,
            weight_decay=CFG.WEIGHT_DECAY,
            gradient_accumulation_steps=CFG.GRAD_ACC,
            warmup_ratio=CFG.WARMUP_RATIO,
            logging_strategy="epoch",
            eval_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            report_to="none",
            fp16=torch.cuda.is_available(),
            dataloader_num_workers=4,
            disable_tqdm=False,
            seed=seed,
        )
        
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_split,
            eval_dataset=val_split,
            processing_class=tokenizer,  # Remplace tokenizer= (deprecated)
            data_collator=collator,
            compute_metrics=compute_metrics,
        )
        
        trainer.train()
        
        # OOF predictions
        val_logits = trainer.predict(val_split).predictions
        val_prob = softmax(val_logits, axis=1)[:, 1]
        oof_probs[val_idx] = val_prob
        
        # Test predictions
        test_logits = trainer.predict(test_tokenized).predictions
        test_probs += softmax(test_logits, axis=1)
        
        # Fold accuracy
        fold_acc = accuracy_score(labels[val_idx], (val_prob >= 0.5).astype(int))
        if verbose:
            print(f"   Fold {fold+1} accuracy: {fold_acc:.4f}")
        
        # 🧹 Libérer mémoire de manière agressive
        del model, trainer, train_split, val_split, val_logits, test_logits
        clean_memory(verbose=verbose)
        
        # Supprimer les fichiers temporaires du fold
        clean_tmp_folder(str(out_dir))
    
    # Moyenne des folds
    test_probs /= CFG.N_SPLITS
    
    return oof_probs, test_probs

In [9]:
# =============================================================================
# ENTRAÎNEMENT MULTI-SEED
# =============================================================================
print("🚀 Démarrage entraînement multi-seed...")
print_memory_status()

# Nettoyage préalable
clean_tmp_folder("/tmp/transformer_ultimate")
clean_memory(verbose=True)

all_oof_probs = []
all_test_probs = []

for seed in CFG.SEEDS:
    oof, test = train_transformer_cv(seed)
    all_oof_probs.append(oof)
    all_test_probs.append(test)
    
    # Nettoyage après chaque seed
    clean_tmp_folder(f"/tmp/transformer_ultimate/seed{seed}")
    clean_memory(verbose=True)

# Moyenne des seeds
transformer_oof = np.mean(all_oof_probs, axis=0)
transformer_test = np.mean([p[:, 1] for p in all_test_probs], axis=0)

# Optimisation du seuil sur OOF moyenné
thresholds = np.linspace(0.35, 0.65, 31)
best_thr_tf = 0.5
best_acc_tf = 0

for thr in thresholds:
    acc = accuracy_score(labels, (transformer_oof >= thr).astype(int))
    if acc > best_acc_tf:
        best_acc_tf = acc
        best_thr_tf = thr

print(f"\n🎯 Transformer Multi-Seed OOF Accuracy: {best_acc_tf:.4f} @ thr={best_thr_tf:.3f}")
print_memory_status()

# Sauvegarder les prédictions
np.save(MODEL_DIR / "transformer_oof.npy", transformer_oof)
np.save(MODEL_DIR / "transformer_test.npy", transformer_test)

🚀 Démarrage entraînement multi-seed...
📊 GPU Memory: 0.00GB / 20.97GB (reserved: 0.00GB)
   🧹 GPU: 0.00GB allocated, 0.00GB reserved

🌱 SEED = 42
   🧹 GPU: 0.00GB allocated, 0.00GB reserved

🌱 SEED = 42

----- Fold 1/5 -----

----- Fold 1/5 -----


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 🌲 Phase 2: XGBoost GPU Renforcé

In [ ]:
def train_xgb_cv(seed: int = 42, verbose: bool = True) -> Tuple[np.ndarray, np.ndarray]:
    """Entraîne XGBoost en CV avec paramètres GPU optimisés.
    
    Returns:
        oof_probs: Out-of-fold predictions
        test_probs: Predictions moyennées
    """
    set_seed(seed)
    if verbose:
        print(f"\n{'='*60}")
        print(f"🌲 XGBoost GPU - SEED={seed}")
        print(f"   max_depth={CFG.XGB_MAX_DEPTH}, n_estimators={CFG.XGB_N_ESTIMATORS}")
        print(f"{'='*60}")
    
    skf = StratifiedGroupKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=seed)
    
    oof_probs = np.zeros(len(labels))
    test_probs = np.zeros(len(X_test_feat))
    
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_feat, labels, groups)):
        if verbose:
            print(f"\n----- Fold {fold+1}/{CFG.N_SPLITS} -----")
        
        clf = XGBClassifier(
            n_estimators=CFG.XGB_N_ESTIMATORS,
            learning_rate=CFG.XGB_LR,
            max_depth=CFG.XGB_MAX_DEPTH,
            min_child_weight=CFG.XGB_MIN_CHILD_WEIGHT,
            subsample=CFG.XGB_SUBSAMPLE,
            colsample_bytree=CFG.XGB_COLSAMPLE,
            reg_alpha=0.05,
            reg_lambda=0.2,
            gamma=0.1,
            tree_method="hist",
            device="cuda",
            # predictor supprimé - n'est plus utilisé avec device="cuda"
            eval_metric="logloss",
            random_state=seed,
            max_bin=512,
            early_stopping_rounds=100,
        )
        
        clf.fit(
            X_feat[tr_idx], labels[tr_idx],
            eval_set=[(X_feat[val_idx], labels[val_idx])],
            verbose=False
        )
        
        oof_probs[val_idx] = clf.predict_proba(X_feat[val_idx])[:, 1]
        test_probs += clf.predict_proba(X_test_feat)[:, 1]
        
        fold_acc = accuracy_score(labels[val_idx], (oof_probs[val_idx] >= 0.5).astype(int))
        if verbose:
            print(f"   Fold {fold+1} accuracy: {fold_acc:.4f} (best_iter={clf.best_iteration})")
    
    test_probs /= CFG.N_SPLITS
    return oof_probs, test_probs

In [ ]:
# Entraîner XGBoost
xgb_oof, xgb_test = train_xgb_cv(seed=42)

# Optimisation du seuil
best_thr_xgb = 0.5
best_acc_xgb = 0
for thr in thresholds:
    acc = accuracy_score(labels, (xgb_oof >= thr).astype(int))
    if acc > best_acc_xgb:
        best_acc_xgb = acc
        best_thr_xgb = thr

print(f"\n🌲 XGBoost OOF Accuracy: {best_acc_xgb:.4f} @ thr={best_thr_xgb:.3f}")

np.save(MODEL_DIR / "xgb_oof.npy", xgb_oof)
np.save(MODEL_DIR / "xgb_test.npy", xgb_test)

## 🔗 Phase 3: Méta-Ensemble avec Logistic Regression

In [ ]:
# =============================================================================
# MÉTA-LOGISTIC REGRESSION SUR OOF
# =============================================================================
print("\n" + "="*60)
print("📊 Méta-Ensemble: LogisticRegression sur OOF")
print("="*60)

# Stack des probas OOF
stack_X = np.vstack([transformer_oof, xgb_oof]).T
stack_test = np.vstack([transformer_test, xgb_test]).T

print(f"Stack shape: {stack_X.shape}")

# LogisticRegression avec calibration
meta = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
meta.fit(stack_X, labels)

meta_oof = meta.predict_proba(stack_X)[:, 1]
meta_test = meta.predict_proba(stack_test)[:, 1]

# Optimisation du seuil
best_thr_meta = 0.5
best_acc_meta = 0
for thr in thresholds:
    acc = accuracy_score(labels, (meta_oof >= thr).astype(int))
    if acc > best_acc_meta:
        best_acc_meta = acc
        best_thr_meta = thr

print(f"\n🎯 Meta-LR OOF Accuracy: {best_acc_meta:.4f} @ thr={best_thr_meta:.3f}")
print(f"   Coefficients: {meta.coef_[0]}")
print(f"   Intercept: {meta.intercept_[0]:.4f}")

In [ ]:
# =============================================================================
# RECHERCHE OPTIMALE DE POIDS POUR BLEND SIMPLE
# =============================================================================
print("\n" + "="*60)
print("⚖️ Recherche du meilleur blend (w * Transformer + (1-w) * XGB)")
print("="*60)

best_w = 0.5
best_thr_blend = 0.5
best_acc_blend = 0

for w in np.linspace(0, 1, 21):
    blended_oof = w * transformer_oof + (1 - w) * xgb_oof
    for thr in thresholds:
        acc = accuracy_score(labels, (blended_oof >= thr).astype(int))
        if acc > best_acc_blend:
            best_acc_blend = acc
            best_w = w
            best_thr_blend = thr

print(f"\n✅ Best Blend: w={best_w:.2f} (Transformer), thr={best_thr_blend:.3f}")
print(f"   Accuracy: {best_acc_blend:.4f}")

# Appliquer le blend au test
blend_test = best_w * transformer_test + (1 - best_w) * xgb_test

## 🔬 Phase 3.5: Techniques Avancées (Conditionnelles)

In [ ]:
# =============================================================================
# 🔬 ADVERSARIAL VALIDATION - Détection du shift train/test
# =============================================================================
print("\n" + "="*70)
print("🔬 ADVERSARIAL VALIDATION")
print("="*70)

# Créer un dataset pour distinguer train vs test
adv_X = np.vstack([X_feat, X_test_feat])
adv_y = np.array([0] * len(X_feat) + [1] * len(X_test_feat))  # 0=train, 1=test

# Shuffle
shuffle_idx = np.random.permutation(len(adv_y))
adv_X, adv_y = adv_X[shuffle_idx], adv_y[shuffle_idx]

# Entraîner un LightGBM rapide
adv_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    n_jobs=1,
    random_state=42,
    verbose=-1
)

# CV pour mesurer l'AUC
adv_probs = cross_val_predict(adv_model, adv_X, adv_y, cv=3, method='predict_proba')[:, 1]
adv_auc = roc_auc_score(adv_y, adv_probs)

print(f"\n📊 Adversarial AUC: {adv_auc:.4f}")

if adv_auc > 0.55:
    print(f"   ⚠️ Shift détecté! Train et test sont distinguables (AUC > 0.55)")
    print(f"   → Les features les plus discriminantes entre train/test peuvent causer de l'overfitting")
    
    # Identifier les features problématiques
    adv_model.fit(adv_X, adv_y)
    feat_importance = adv_model.feature_importances_
    top_adv_features = np.argsort(feat_importance)[-5:][::-1]
    print(f"   → Top features discriminantes (indices): {top_adv_features}")
    
    # Option: downweight les samples train "trop différents" du test
    # On ne va PAS supprimer des features car elles peuvent quand même être utiles
    ADVERSARIAL_SHIFT_DETECTED = True
else:
    print(f"   ✅ Pas de shift significatif (AUC ≈ 0.5)")
    ADVERSARIAL_SHIFT_DETECTED = False

del adv_X, adv_y, adv_model, adv_probs
gc.collect()

In [ ]:
# =============================================================================
# 🎯 FEATURE SELECTION - Garder seulement les features utiles
# =============================================================================
print("\n" + "="*70)
print("🎯 FEATURE SELECTION (basée sur XGBoost importance)")
print("="*70)

# Entraîner un XGBoost rapide pour obtenir les importances
fs_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    tree_method="hist",
    device="cuda",
    random_state=42,
    early_stopping_rounds=50,
)

# Split simple pour feature importance
X_tr, X_val, y_tr, y_val = train_test_split(X_feat, labels, test_size=0.2, stratify=labels, random_state=42)
fs_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

# Importance des features
importances = fs_model.feature_importances_
n_features = len(importances)

# Tester différents seuils de sélection
print(f"\n📊 Analyse de {n_features} features...")

best_n_features = n_features
best_acc_fs = best_acc_xgb  # Accuracy XGB de base

for keep_pct in [0.9, 0.8, 0.7, 0.6, 0.5]:
    n_keep = int(n_features * keep_pct)
    top_idx = np.argsort(importances)[-n_keep:]
    
    # Évaluer avec les features sélectionnées
    X_fs = X_feat[:, top_idx]
    X_test_fs = X_test_feat[:, top_idx]
    
    # CV rapide
    skf_fs = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    oof_fs = np.zeros(len(labels))
    
    for tr_idx, val_idx in skf_fs.split(X_fs, labels):
        clf_fs = XGBClassifier(
            n_estimators=500, max_depth=8, learning_rate=0.05,
            tree_method="hist", device="cuda", random_state=42
        )
        clf_fs.fit(X_fs[tr_idx], labels[tr_idx], verbose=False)
        oof_fs[val_idx] = clf_fs.predict_proba(X_fs[val_idx])[:, 1]
    
    acc_fs = accuracy_score(labels, (oof_fs >= 0.5).astype(int))
    print(f"   {keep_pct*100:.0f}% features ({n_keep}): accuracy = {acc_fs:.4f}")
    
    if acc_fs > best_acc_fs + 0.001:  # Amélioration significative
        best_acc_fs = acc_fs
        best_n_features = n_keep
        best_feature_idx = top_idx

# Appliquer la sélection si bénéfique
if best_n_features < n_features:
    print(f"\n✅ Feature selection appliquée: {n_features} → {best_n_features} features")
    print(f"   Gain accuracy: +{(best_acc_fs - best_acc_xgb)*100:.2f}%")
    X_feat_selected = X_feat[:, best_feature_idx]
    X_test_feat_selected = X_test_feat[:, best_feature_idx]
    FEATURE_SELECTION_APPLIED = True
else:
    print(f"\n⏭️ Feature selection non bénéfique, on garde toutes les features")
    X_feat_selected = X_feat
    X_test_feat_selected = X_test_feat
    FEATURE_SELECTION_APPLIED = False

del fs_model, X_tr, X_val, y_tr, y_val
gc.collect()

In [ ]:
# =============================================================================
# 📐 CALIBRATION ISOTONIQUE - Améliorer les probabilités
# =============================================================================
print("\n" + "="*70)
print("📐 CALIBRATION ISOTONIQUE")
print("="*70)

# Sauvegarder les probas originales
meta_oof_original = meta_oof.copy()
meta_test_original = meta_test.copy()

# Calibration isotonique sur les probas meta
ir = IsotonicRegression(out_of_bounds='clip')
ir.fit(meta_oof, labels)

# Appliquer la calibration
meta_oof_calibrated = ir.predict(meta_oof)
meta_test_calibrated = ir.predict(meta_test)

# Évaluer l'amélioration
best_acc_calib = 0
best_thr_calib = 0.5

for thr in thresholds:
    acc = accuracy_score(labels, (meta_oof_calibrated >= thr).astype(int))
    if acc > best_acc_calib:
        best_acc_calib = acc
        best_thr_calib = thr

print(f"\n📊 Comparaison:")
print(f"   Meta-LR original:  {best_acc_meta:.4f} @ thr={best_thr_meta:.3f}")
print(f"   Meta-LR calibré:   {best_acc_calib:.4f} @ thr={best_thr_calib:.3f}")

# Appliquer seulement si amélioration
if best_acc_calib > best_acc_meta:
    improvement = (best_acc_calib - best_acc_meta) * 100
    print(f"\n✅ Calibration appliquée! Gain: +{improvement:.2f}%")
    meta_oof_final = meta_oof_calibrated
    meta_test_final = meta_test_calibrated
    best_acc_meta_final = best_acc_calib
    best_thr_meta_final = best_thr_calib
    CALIBRATION_APPLIED = True
else:
    print(f"\n⏭️ Calibration non bénéfique, on garde les probas originales")
    meta_oof_final = meta_oof
    meta_test_final = meta_test
    best_acc_meta_final = best_acc_meta
    best_thr_meta_final = best_thr_meta
    CALIBRATION_APPLIED = False

# Calibration aussi sur Transformer seul
ir_tf = IsotonicRegression(out_of_bounds='clip')
ir_tf.fit(transformer_oof, labels)
transformer_oof_calibrated = ir_tf.predict(transformer_oof)
transformer_test_calibrated = ir_tf.predict(transformer_test)

best_acc_tf_calib = 0
best_thr_tf_calib = 0.5
for thr in thresholds:
    acc = accuracy_score(labels, (transformer_oof_calibrated >= thr).astype(int))
    if acc > best_acc_tf_calib:
        best_acc_tf_calib = acc
        best_thr_tf_calib = thr

print(f"\n   Transformer original: {best_acc_tf:.4f} @ thr={best_thr_tf:.3f}")
print(f"   Transformer calibré:  {best_acc_tf_calib:.4f} @ thr={best_thr_tf_calib:.3f}")

if best_acc_tf_calib > best_acc_tf:
    transformer_oof_final = transformer_oof_calibrated
    transformer_test_final = transformer_test_calibrated
    best_acc_tf_final = best_acc_tf_calib
    best_thr_tf_final = best_thr_tf_calib
    print(f"   ✅ Calibration Transformer appliquée!")
else:
    transformer_oof_final = transformer_oof
    transformer_test_final = transformer_test
    best_acc_tf_final = best_acc_tf
    best_thr_tf_final = best_thr_tf

## 🏷️ Phase 4: Pseudo-Labeling (Optionnel)

In [ ]:
# =============================================================================
# PSEUDO-LABELING AVEC LES PRÉDICTIONS HAUTE CONFIANCE
# =============================================================================
ENABLE_PSEUDO_LABELING = True  # Mettre à False pour skip

if ENABLE_PSEUDO_LABELING:
    print("\n" + "="*60)
    print("🏷️ Pseudo-Labeling sur le Test Set")
    print("="*60)
    
    # Utiliser les prédictions méta (calibrées si applicable) pour sélectionner les pseudo-labels
    pseudo_probs = meta_test_final if 'meta_test_final' in dir() else meta_test
    
    # Analyse de la distribution des probas test
    print(f"\n📊 Distribution des probas test (meta):")
    print(f"   Quantiles: 5%={np.percentile(pseudo_probs, 5):.3f}, "
          f"25%={np.percentile(pseudo_probs, 25):.3f}, "
          f"50%={np.percentile(pseudo_probs, 50):.3f}, "
          f"75%={np.percentile(pseudo_probs, 75):.3f}, "
          f"95%={np.percentile(pseudo_probs, 95):.3f}")
    
    # Tester plusieurs seuils pour choisir le meilleur
    thresholds_pseudo = [
        (0.05, 0.95, "Très conservateur"),
        (0.10, 0.90, "Conservateur"),
        (0.15, 0.85, "Modéré (défaut)"),
        (0.20, 0.80, "Agressif"),
    ]
    
    print(f"\n📋 Analyse des seuils de pseudo-labeling:")
    for thr_low, thr_high, label in thresholds_pseudo:
        n_high = (pseudo_probs >= thr_high).sum()
        n_low = (pseudo_probs <= thr_low).sum()
        total = n_high + n_low
        pct = total / len(pseudo_probs) * 100
        print(f"   {label:20s}: {total:,} samples ({pct:.1f}%) - "
              f"Inf={n_high:,}, Obs={n_low:,}")
    
    # Utiliser les seuils de la config
    high_conf_1 = pseudo_probs >= CFG.PSEUDO_THR_HIGH  # Probables Influencers
    high_conf_0 = pseudo_probs <= CFG.PSEUDO_THR_LOW   # Probables Observers
    
    n_pseudo_1 = high_conf_1.sum()
    n_pseudo_0 = high_conf_0.sum()
    
    print(f"\n🎯 Seuils sélectionnés: [{CFG.PSEUDO_THR_LOW}, {CFG.PSEUDO_THR_HIGH}]")
    print(f"   Pseudo-labels Influencer (p≥{CFG.PSEUDO_THR_HIGH}): {n_pseudo_1:,}")
    print(f"   Pseudo-labels Observer (p≤{CFG.PSEUDO_THR_LOW}): {n_pseudo_0:,}")
    print(f"   Total pseudo-labels: {n_pseudo_1 + n_pseudo_0:,} / {len(test_raw):,} "
          f"({(n_pseudo_1 + n_pseudo_0)/len(test_raw)*100:.1f}%)")
    
    # Vérifier l'équilibre des classes
    if n_pseudo_1 + n_pseudo_0 > 500:
        balance = n_pseudo_1 / (n_pseudo_1 + n_pseudo_0)
        print(f"   Balance Inf/Total: {balance:.2%}")
        
        if 0.3 <= balance <= 0.7:
            print("   ✅ Classes pseudo-labels relativement équilibrées")
        else:
            print("   ⚠️ Classes déséquilibrées, attention au biais!")
    
    if n_pseudo_1 + n_pseudo_0 > 500:  # Seuil minimum abaissé
        # Créer le dataset augmenté
        pseudo_idx_1 = np.where(high_conf_1)[0]
        pseudo_idx_0 = np.where(high_conf_0)[0]
        
        pseudo_df = pd.concat([
            test_raw.iloc[pseudo_idx_1].assign(label=1),
            test_raw.iloc[pseudo_idx_0].assign(label=0),
        ])
        
        # Fusionner avec train
        train_augmented = pd.concat([train_raw, pseudo_df], ignore_index=True)
        
        print(f"\n📈 Augmentation des données:")
        print(f"   Train original: {len(train_raw):,}")
        print(f"   + Pseudo-labels: {len(pseudo_df):,}")
        print(f"   = Train augmenté: {len(train_augmented):,}")
        
        # Nouvelle distribution des labels
        aug_label_dist = train_augmented["label"].value_counts(normalize=True)
        print(f"   Distribution augmentée: Obs={aug_label_dist.get(0, 0):.2%}, Inf={aug_label_dist.get(1, 0):.2%}")
        
        # Recréer les labels et groupes
        labels_aug = train_augmented["label"].astype(int).to_numpy()
        groups_aug = train_augmented["group"].to_numpy()
        
        # Retokeniser
        train_aug_ds = Dataset.from_pandas(train_augmented[["text_input", "label", "group"]])
        aug_remove_cols = [c for c in ["text_input", "group", "__index_level_0__"] if c in train_aug_ds.column_names]
        train_aug_tokenized = train_aug_ds.map(tokenize_fn, batched=True, remove_columns=aug_remove_cols)
        
        print("\n🔄 Dataset augmenté prêt pour le re-entraînement...")
    else:
        print("\n⚠️ Pas assez de pseudo-labels haute confiance, skip.")
        ENABLE_PSEUDO_LABELING = False

In [ ]:
# =============================================================================
# RE-ENTRAÎNEMENT AVEC PSEUDO-LABELS
# =============================================================================
# Initialiser pseudo_test_probs au cas où le pseudo-labeling est skip
pseudo_test_probs = None

if ENABLE_PSEUDO_LABELING and 'n_pseudo_1' in dir() and (n_pseudo_1 + n_pseudo_0 > 500):
    set_seed(42)
    
    print("\n" + "="*60)
    print("🔄 Re-entraînement Transformer avec Pseudo-Labels")
    print("="*60)
    print_memory_status()
    
    # Nettoyage préalable
    clean_tmp_folder("/tmp/transformer_pseudo")
    clean_memory()
    
    # Entraînement sur le dataset augmenté (sans CV pour gagner du temps)
    model_pseudo = AutoModelForSequenceClassification.from_pretrained(
        CFG.MODEL_NAME, 
        num_labels=2,
        ignore_mismatched_sizes=True
    )
    
    args_pseudo = TrainingArguments(
        output_dir="/tmp/transformer_pseudo",
        num_train_epochs=CFG.PSEUDO_EPOCHS,
        per_device_train_batch_size=CFG.BATCH_SIZE,
        per_device_eval_batch_size=CFG.BATCH_SIZE * 2,
        learning_rate=CFG.PSEUDO_LR,  # LR plus faible pour affinage
        weight_decay=CFG.WEIGHT_DECAY,
        gradient_accumulation_steps=CFG.GRAD_ACC,
        warmup_ratio=0.05,
        logging_strategy="epoch",
        save_strategy="no",
        report_to="none",
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=4,
        seed=42,
    )
    
    trainer_pseudo = Trainer(
        model=model_pseudo,
        args=args_pseudo,
        train_dataset=train_aug_tokenized,
        processing_class=tokenizer,  # Remplace tokenizer= (deprecated)
        data_collator=collator,
    )
    
    print(f"   Epochs: {CFG.PSEUDO_EPOCHS}, LR: {CFG.PSEUDO_LR}")
    trainer_pseudo.train()
    
    # Nouvelles prédictions sur le test
    print("\n📊 Génération des prédictions pseudo-labeling...")
    pseudo_test_logits = trainer_pseudo.predict(test_tokenized).predictions
    pseudo_test_probs = softmax(pseudo_test_logits, axis=1)[:, 1]
    
    # Évaluation (comparaison avec meta_test)
    print(f"\n📈 Comparaison des distributions:")
    print(f"   Meta test probs: mean={meta_test.mean():.3f}, std={meta_test.std():.3f}")
    print(f"   Pseudo test probs: mean={pseudo_test_probs.mean():.3f}, std={pseudo_test_probs.std():.3f}")
    
    # Corrélation entre les deux
    corr = np.corrcoef(meta_test, pseudo_test_probs)[0, 1]
    print(f"   Corrélation meta/pseudo: {corr:.4f}")
    
    # Sauvegarder les prédictions
    np.save(MODEL_DIR / "pseudo_test.npy", pseudo_test_probs)
    
    print(f"\n✅ Pseudo-labeling terminé!")
    
    # 🧹 Cleanup agressif
    del model_pseudo, trainer_pseudo, pseudo_test_logits
    del train_aug_tokenized, train_aug_ds, train_augmented, pseudo_df
    clean_tmp_folder("/tmp/transformer_pseudo")
    clean_memory(verbose=True)
else:
    print("\n⏭️ Pseudo-labeling désactivé ou pas assez de données.")

## 📤 Phase 5: Génération des Soumissions

In [ ]:
# =============================================================================
# RÉCAPITULATIF DES RÉSULTATS
# =============================================================================
print("\n" + "="*70)
print("📊 RÉCAPITULATIF DES RÉSULTATS")
print("="*70)

# Résultats de base
results = {
    "Transformer Multi-Seed": (best_acc_tf, best_thr_tf),
    "XGBoost GPU": (best_acc_xgb, best_thr_xgb),
    "Meta-LR": (best_acc_meta, best_thr_meta),
    "Blend Optimal": (best_acc_blend, best_thr_blend),
}

# Ajouter les résultats calibrés si disponibles
if 'best_acc_tf_final' in dir() and best_acc_tf_final != best_acc_tf:
    results["Transformer Calibré"] = (best_acc_tf_final, best_thr_tf_final)
if 'best_acc_meta_final' in dir() and best_acc_meta_final != best_acc_meta:
    results["Meta-LR Calibré"] = (best_acc_meta_final, best_thr_meta_final)

for name, (acc, thr) in results.items():
    marker = "⭐" if "Calibré" in name else "  "
    print(f" {marker} {name:25s} : {acc:.4f} @ thr={thr:.3f}")

# Résumé des techniques appliquées
print("\n📋 Techniques avancées:")
if 'ADVERSARIAL_SHIFT_DETECTED' in dir():
    print(f"   Adversarial Validation: {'⚠️ Shift détecté' if ADVERSARIAL_SHIFT_DETECTED else '✅ OK'}")
if 'FEATURE_SELECTION_APPLIED' in dir():
    print(f"   Feature Selection: {'✅ Appliquée' if FEATURE_SELECTION_APPLIED else '⏭️ Non bénéfique'}")
if 'CALIBRATION_APPLIED' in dir():
    print(f"   Calibration Isotonique: {'✅ Appliquée' if CALIBRATION_APPLIED else '⏭️ Non bénéfique'}")

# Meilleur modèle
best_model = max(results.items(), key=lambda x: x[1][0])
print(f"\n🏆 Meilleur modèle: {best_model[0]} ({best_model[1][0]:.4f})")

In [ ]:
# =============================================================================
# GÉNÉRATION DES SOUMISSIONS
# =============================================================================
print("\n📤 Génération des fichiers de soumission...")

submissions = {}

# Utiliser les probas finales (calibrées si applicable)
tf_test_final = transformer_test_final if 'transformer_test_final' in dir() else transformer_test
tf_thr_final = best_thr_tf_final if 'best_thr_tf_final' in dir() else best_thr_tf
meta_test_to_use = meta_test_final if 'meta_test_final' in dir() else meta_test
meta_thr_to_use = best_thr_meta_final if 'best_thr_meta_final' in dir() else best_thr_meta

# 1. Transformer seul (différents seuils)
for thr in [0.5, 0.55, tf_thr_final]:
    pred = (tf_test_final >= thr).astype(int)
    name = f"submission_tf_thr{thr:.2f}.csv"
    pd.DataFrame({"ID": test_ids, "Prediction": pred}).to_csv(
        SUBMISSION_DIR / name, index=False)
    submissions[name] = thr

# 2. XGBoost seul
pred_xgb = (xgb_test >= best_thr_xgb).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": pred_xgb}).to_csv(
    SUBMISSION_DIR / "submission_xgb_ultimate.csv", index=False)
submissions["submission_xgb_ultimate.csv"] = best_thr_xgb

# 3. Meta-LR (calibré si applicable) - RECOMMANDÉ
pred_meta = (meta_test_to_use >= meta_thr_to_use).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": pred_meta}).to_csv(
    SUBMISSION_DIR / "submission_meta_ultimate.csv", index=False)
submissions["submission_meta_ultimate.csv"] = meta_thr_to_use

# 4. Blend optimal
pred_blend = (blend_test >= best_thr_blend).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": pred_blend}).to_csv(
    SUBMISSION_DIR / "submission_blend_ultimate.csv", index=False)
submissions["submission_blend_ultimate.csv"] = best_thr_blend

# 5. Calibré si différent de l'original
if 'CALIBRATION_APPLIED' in dir() and CALIBRATION_APPLIED:
    pred_meta_calib = (meta_test_final >= best_thr_meta_final).astype(int)
    pd.DataFrame({"ID": test_ids, "Prediction": pred_meta_calib}).to_csv(
        SUBMISSION_DIR / "submission_meta_calibrated.csv", index=False)
    submissions["submission_meta_calibrated.csv"] = best_thr_meta_final

# 6. Pseudo-labeling (si activé)
if 'pseudo_test_probs' in dir() and pseudo_test_probs is not None:
    # Essayer différents seuils
    for thr in [0.45, 0.50, 0.55]:
        pred_pseudo = (pseudo_test_probs >= thr).astype(int)
        name = f"submission_pseudo_thr{thr:.2f}.csv"
        pd.DataFrame({"ID": test_ids, "Prediction": pred_pseudo}).to_csv(
            SUBMISSION_DIR / name, index=False)
        submissions[name] = thr
    
    # Ensemble: moyenne meta + pseudo
    ensemble_pseudo = (meta_test_to_use + pseudo_test_probs) / 2
    best_thr_ens_pseudo = 0.5
    pred_ens_pseudo = (ensemble_pseudo >= best_thr_ens_pseudo).astype(int)
    pd.DataFrame({"ID": test_ids, "Prediction": pred_ens_pseudo}).to_csv(
        SUBMISSION_DIR / "submission_meta_pseudo_ensemble.csv", index=False)
    submissions["submission_meta_pseudo_ensemble.csv"] = best_thr_ens_pseudo

print("\n✅ Fichiers générés:")
for name, thr in submissions.items():
    print(f"   📄 {name} (thr={thr:.3f})")

In [ ]:
# =============================================================================
# ANALYSE DES PRÉDICTIONS
# =============================================================================
print("\n" + "="*70)
print("📊 ANALYSE DES PRÉDICTIONS")
print("="*70)

# Distribution des probas
print(f"\nTransformer test probs: min={transformer_test.min():.3f}, max={transformer_test.max():.3f}, mean={transformer_test.mean():.3f}")
print(f"XGBoost test probs: min={xgb_test.min():.3f}, max={xgb_test.max():.3f}, mean={xgb_test.mean():.3f}")
print(f"Meta test probs: min={meta_test.min():.3f}, max={meta_test.max():.3f}, mean={meta_test.mean():.3f}")

# Proportion de predictions positives
print(f"\nProportion Influencer dans prédictions:")
print(f"   Transformer: {(transformer_test >= best_thr_tf).mean():.3f}")
print(f"   XGBoost: {(xgb_test >= best_thr_xgb).mean():.3f}")
print(f"   Meta: {(meta_test >= best_thr_meta).mean():.3f}")
print(f"   Train original: {labels.mean():.3f}")

In [ ]:
# =============================================================================
# SOUMISSION RECOMMANDÉE
# =============================================================================
print("\n" + "="*70)
print("🎯 RECOMMANDATION DE SOUMISSION")
print("="*70)

print("""
Ordre de priorité pour soumettre sur Kaggle:

1. 🥇 submission_meta_ultimate.csv  (Meta-LR - combine TF + XGB)
2. 🥈 submission_blend_ultimate.csv (Blend simple optimisé)
3. 🥉 submission_tf_thr{best_thr_tf:.2f}.csv  (Transformer seul)

Si l'écart CV/LB est trop grand, essayer:
- Réduire les epochs (overfitting)
- Augmenter le seuil vers 0.55-0.60
- Vérifier la cohérence des groupes
""")

print(f"\n📁 Tous les fichiers dans: {SUBMISSION_DIR}")

## 📝 Notes et Améliorations Implémentées

### Ce qui a été implémenté:
- ✅ Multi-seed (42, 1234, 2024, 0) avec moyenne des probas
- ✅ MAX_LEN=224 et EPOCHS=5, N_SPLITS=5
- ✅ XGBoost GPU avec max_depth=10, n_estimators=2000
- ✅ Méta-LogisticRegression sur OOF
- ✅ Recherche de seuil optimale
- ✅ Pseudo-labeling (optionnel)
- ✅ **Adversarial Validation** : Détecte si train/test sont différents
- ✅ **Feature Selection** : Garde seulement les features qui améliorent l'accuracy
- ✅ **Calibration Isotonique** : Améliore les probabilités (appliquée ssi bénéfique)

### Logique conditionnelle:
Chaque technique avancée n'est appliquée **que si elle améliore l'accuracy** sur les OOF predictions. Sinon, on garde les prédictions originales.